# ResNet18 development on CPU
Use ImageNet transfer learning and one output logit. The first run downloads the pretrained weight file; subsequent runs reuse its local cache.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "configs/cpu.yaml").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks directory")
os.chdir(ROOT)
import yaml
from ml.runtime import configure
config = yaml.safe_load(Path("configs/cpu.yaml").read_text())
configure(config["seed"], config["threads"])

In [ ]:
import torch
from ml.models.resnet import build_model, set_stage
model = build_model(pretrained=True)
set_stage(model, "head")
model.eval()
with torch.no_grad():
    logits = model(torch.zeros(8, 3, config["image_size"], config["image_size"]))
assert logits.shape == (8, 1)
{"output_shape": tuple(logits.shape),
 "parameter_memory_mib": sum(p.numel()*p.element_size() for p in model.parameters()) / 1024**2,
 "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
 "device": str(next(model.parameters()).device)}

## Training strategy
Stage A trains the classifier with a frozen backbone and frozen BatchNorm statistics. Optional Stage B unfreezes layer4 with a smaller learning rate. BCEWithLogitsLoss uses a training-only positive-class weight; no weighted sampler is combined with it. Parameter memory above excludes activations and optimizer state.

In [ ]:
model.fc